## Name: Sanyam Goel

# Embedding matrix

### Assignment 5

This class cretaes an Embedding matrix

* Seed of random number generator has to be passed to EmbeddingMatrix constructor while creating class object, based on this seed a random number generator is initialized as a field.
*  #### create_embedding_matrix
  This method creates an embedding matrix

In [1]:
"""This exception is raised when an invalid range is passed to embedding matrix generation method"""
class InvalidRangeException(Exception):

    def __init__(self, message):
        super().__init__(message)

In [2]:
"""This exception is raised when the probability of rows does not equal 1 """
class ProbabilityNotOneException(Exception):

    def __init__(self, message):
        super().__init__(message)

In [3]:
"""This exception is raised when invalid matrix dimension are passed to embedding matrix generation method"""
class InvalidDimensionException(Exception):

    def __init__(self, message):
        super().__init__(message)

In [4]:
""" Module for embedding matrix functionality """
from numpy import random
import numpy as np
import numpy.typing as npt 
from typing import Tuple

class EmbeddingMatrix:
    
    def __init__(self, seed:int):

        self.random_num_generator = random.default_rng(seed)
        pass

    def create_embedding_matrix(self, matrix_dimension:Tuple[int,int], matrix_values_range:Tuple[int,int], total_zeros:np.int64, embedding_matrix_dimension:Tuple[int,int], output_dimension:int) -> npt.NDArray[np.float64]:
        
        """ Create embedding matrix by generating 
        random matrix based on passed dimesnions and range"""
        
        row_size, column_size = matrix_dimension

        range_low_value, range_high_value = matrix_values_range
        
        self._validate_range(range_low_value, range_high_value)

        self._validate_dimensions(row_size, column_size)

        # generates words matrix represented by vectors
        generated_word_matrix = self.random_num_generator.integers(low=range_low_value, high= range_high_value,size=(row_size, column_size))

         # Find random indices, to be replaced by zeros
        row_indices, column_indices = self._generate_random_indexes(matrix_dimension, total_zeros)
        
        generated_word_matrix[row_indices,column_indices] = 0

        # Create embedding matrix
        self.embedding_matrix = self.random_num_generator.uniform(size=(embedding_matrix_dimension[0], embedding_matrix_dimension[1]))

        # Create Zero row representing non matching word in embedding matrix
        zeros_row= np.zeros(embedding_matrix_dimension[1])

        #stack zeros as top row in the embedding matrix to represent non matcing word
        self.embedding_matrix= np.vstack([zeros_row,self.embedding_matrix])

        #select vectors from embedding matrix based on gnerated_word_matrix values/indices
        matrix_x= self.embedding_matrix[generated_word_matrix]

        mask = generated_word_matrix != 0

        #expand with new axis
        mask_expanded = mask[..., np.newaxis] 

        matrix_x_masked= matrix_x * mask_expanded

        sum_embeddings = matrix_x_masked.sum(axis=1)
        
        count_nonzero = mask.sum(axis=1).reshape(-1, 1)

        x_avg = np.divide(sum_embeddings, count_nonzero, where=count_nonzero != 0)

        rng_weights = np.random.default_rng(seed=2)
        
        w0 = rng_weights.uniform(-1.0, 1.0, size=(embedding_matrix_dimension[1], output_dimension))  
        
        w1 = rng_weights.uniform(-1.0, 1.0, size=(output_dimension, output_dimension))

        L = np.dot(x_avg, np.dot(w0, w1 ))

        exp_L = np.exp(L - np.max(L, axis=1, keepdims=True))
        
        p = exp_L / exp_L.sum(axis=1, keepdims=True)

        if self._validate_generated_probability_matrix(p):
            return p
        else:
            raise ProbabilityNotOneException('Generated Probability matrix not correct')
            
        
    
    def _validate_range(self, range_low_value:np.int64, 
                               range_high_value:np.int64)-> bool:

        """ This method validates the range"""
        
        if range_low_value<= range_high_value:
            return True
        raise InvalidRangeException(f'Invalid range: ({range_low_value},{range_high_value})')
    
    
    def _validate_dimensions(self, row_size:np.int64, 
                               column_size:np.int64)-> bool:

        """ This method validates the dimesnion of a matrix"""
        
        if row_size < 0 or column_size < 0:
            raise InvalidDimensionException(f'Invalid Dimensions: {row_size}x{column_size}')     

    def _generate_random_indexes(self, matrix_dimension:Tuple[int,int], 
                                 total_zeros:np.int64) -> Tuple[npt.NDArray[np.int64], npt.NDArray[np.int64]]:

        """ This method gerenates random indiexes to be used 
         for zero value insertion on those indices using unravel function"""
        
        row_size, column_size = matrix_dimension
        
        rng= random.default_rng(seed=5)

        selected_indices = rng.choice(a=row_size*column_size, size=total_zeros, replace=False)

        row_indices, column_indices = np.unravel_index(selected_indices, matrix_dimension)        

        return (row_indices, column_indices)

    def _validate_generated_probability_matrix(self, matrix:np.ndarray) -> bool:

        """ This method valdiates all the rows of the 
        generated probability matrix sum to one"""
        
        row_sums = matrix.sum(axis=1)
        
        row_sums_rounded = np.round(row_sums, decimals=7)

        are_all_one_matrix = row_sums_rounded==1

        return np.all(are_all_one_matrix)
        


 ## Test case 1:

In [5]:
emb = EmbeddingMatrix(8)

emb.create_embedding_matrix(matrix_values_range=(1,20), matrix_dimension=(5,10), total_zeros=10, embedding_matrix_dimension=(30,15), output_dimension=5)

array([[0.03971751, 0.04568776, 0.74394462, 0.12726107, 0.04338904],
       [0.03973328, 0.06334507, 0.63719567, 0.18050321, 0.07922277],
       [0.0291483 , 0.05561227, 0.64409907, 0.20155919, 0.06958118],
       [0.04390786, 0.03115919, 0.79285144, 0.0937732 , 0.03830831],
       [0.04848959, 0.06738265, 0.59538788, 0.19147251, 0.09726737]])

 ## Test case 2:

In [6]:
emb.create_embedding_matrix(matrix_values_range=(1,20), matrix_dimension=(10,10), total_zeros=10, embedding_matrix_dimension=(30,15), output_dimension=5)

array([[0.02739076, 0.03301435, 0.71065047, 0.15977509, 0.06916933],
       [0.02109446, 0.00898899, 0.90246583, 0.04710033, 0.02035039],
       [0.03621279, 0.01441377, 0.87128011, 0.05099784, 0.02709548],
       [0.02987722, 0.04084374, 0.68733695, 0.17032257, 0.07161952],
       [0.08100626, 0.03577556, 0.75622634, 0.07311874, 0.05387311],
       [0.02639335, 0.01700477, 0.83460706, 0.08434612, 0.0376487 ],
       [0.03199312, 0.02434552, 0.76813497, 0.10974428, 0.06578211],
       [0.04159491, 0.01696109, 0.82661977, 0.06926561, 0.04555862],
       [0.04841434, 0.04487432, 0.68411065, 0.13476416, 0.08783654],
       [0.02489092, 0.01712621, 0.81604506, 0.09824089, 0.04369693]])

 ## Test case 3:

In [7]:
emb.create_embedding_matrix(matrix_values_range=(1,20), matrix_dimension=(20,10), total_zeros=10, embedding_matrix_dimension=(30,15), output_dimension=15)

array([[2.06801947e-01, 6.54025203e-02, 8.29181513e-04, 6.16911217e-05,
        5.68226947e-04, 8.53117943e-03, 2.69215571e-03, 6.73354059e-06,
        1.02360690e-02, 3.70314698e-01, 3.28179115e-01, 8.95520030e-04,
        1.91097921e-03, 3.61063747e-04, 3.20891922e-03],
       [3.72973751e-01, 1.39032557e-01, 7.43204000e-04, 7.17933517e-05,
        7.18212998e-04, 1.13559040e-02, 4.15360987e-03, 3.63391494e-06,
        8.47799395e-03, 8.82620831e-02, 3.68233044e-01, 8.27908706e-04,
        1.91474667e-03, 3.52730166e-04, 2.87882829e-03],
       [4.62741893e-01, 1.54421001e-01, 7.11974964e-04, 6.04574615e-05,
        1.88814285e-03, 9.02121835e-03, 2.47783680e-03, 2.52907409e-06,
        5.06943257e-03, 2.03299205e-02, 3.38252503e-01, 1.21483585e-03,
        1.11549502e-03, 4.57523877e-04, 2.23523541e-03],
       [2.81792024e-01, 7.38996595e-02, 2.18978971e-03, 1.07435765e-04,
        3.63568740e-03, 7.23953409e-03, 7.53878843e-03, 2.57928841e-05,
        2.82283843e-02, 9.53837373e-0